## 1. Importar librerías

Framework: Pytorch

In [ ]:
!pip install opencv-python
!pip install -q torchinfo
#!pip install -q torch torchvision torchinfo tensorboard scikit-learn seaborn matplotlib numpy pandas opencv-python pillow h5py scipy tqdm

In [ ]:
# Librerías estándar de Python
import os
import json
import gc
import time
import random
import warnings
from glob import glob
from pathlib import Path
from math import sqrt
from datetime import datetime
from typing import Dict, List, Any, Optional, Tuple, Union
from dataclasses import dataclass

# Librerías de manejo de datos y matemáticas
import numpy as np
import pandas as pd
import h5py

# Procesamiento de imágenes
import cv2
from PIL import Image
from scipy.ndimage import gaussian_filter

# Visualización y barras de progreso
import matplotlib.pyplot as plt
import seaborn as sn
from tqdm import tqdm

# Scikit-Learn
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

# Deep Learning: PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim import Adam
from torch.utils.data import DataLoader, Dataset, random_split
from torch.utils.tensorboard import SummaryWriter
from torchinfo import summary

# Deep Learning: Torchvision (Computer Vision)
import torchvision
from torchvision import datasets
from torchvision import models
import torchvision.transforms as T
from torchvision.transforms import v2 as transforms
from torchvision.ops import Conv2dNormActivation

%matplotlib inline
warnings.filterwarnings("ignore")

## 2. Descarga del dataset

Mismo dataset CrowdHuman utilizado en el entrenamiento de YOLO. Esto garantiza que la comparativa CSRNet vs YOLO se realiza sobre el **mismo conjunto de imagenes** de train, validacion y test.

| datos | n imagenes | % |
| --- | --- | --- |
| training y validation | 2552 | 85,04% |
| test | 449 | 14,96% |

In [ ]:
%%bash
wget "https://usales-my.sharepoint.com/:u:/g/personal/hrm_usal_es/IQBwS9aoohFwTLTjjUEdtw7gAYCxn3HUX7rWw0PaKVVNW8M?e=Ny6uQm&download=1" -O "crowdhuman.zip"
mkdir -p data/crowdhuman
unzip -q "crowdhuman.zip" -d data/crowdhuman
cp -r data/crowdhuman/valid/images/* data/crowdhuman/train/images/
cp -r data/crowdhuman/valid/labels/* data/crowdhuman/train/labels/

Estructura de directorios del entorno

In [ ]:
%%bash
echo "Entrenamiento y validacion:" $(ls data/crowdhuman/train/images | wc -l) "imagenes"
echo "Test:" $(ls data/crowdhuman/test/images | wc -l) "imagenes"

In [ ]:
train_root        = os.path.join("data", "crowdhuman", "train")
test_root         = os.path.join("data", "crowdhuman", "test")
train_images_root = os.path.join(train_root, "images")
test_images_root  = os.path.join(test_root,  "images")

# Comprobacion de dimensiones de alguna imagen
for i, file in enumerate(sorted(os.listdir(train_images_root))[:10]):
    path = os.path.join(train_images_root, file)
    with Image.open(path) as img:
        print(f"{file}: {img.size}  (ancho x alto)")

Se mantienen la **media y desviación estandar del backbone VGG-16 (ImageNet)**, mismos valores que en YOLO.

**Carga de anotaciones CrowdHuman: conversión de bounding boxes de cabeza (clase 1, formato YOLO) a puntos centrales para CSRNet**

Las etiquetas de CrowdHuman estan en formato YOLO normalizado `clase cx_norm cy_norm w_norm h_norm`. La clase 1 corresponde a las cabezas. La conversión a punto de anotación para CSRNet es directa:

- `x_px = cx_norm x img_width`
- `y_px = cy_norm x img_height`

El centro geométrico del bounding box de la cabeza equivale al punto de anotación que CSRNet espera para generar el mapa de densidad.

In [ ]:
def cargar_puntos_cabeza_desde_yolo(directorio_raiz: str) -> dict:
    """
    Lee anotaciones en formato YOLO desde directorio_raiz/labels/*.txt,
    filtra la clase 1 (cabeza) y devuelve los centros de las cajas como
    puntos (x, y) en coordenadas de píxel de la imagen original.

    Formato YOLO por línea: clase cx_norm cy_norm w_norm h_norm (normalizados [0,1])
    Conversión a punto:  x_px = cx_norm * img_width,  y_px = cy_norm * img_height

    Args:
        directorio_raiz: carpeta con subcarpetas images/ y labels/

    Devuelve:
        { "nombre_imagen.jpg": [[x1, y1], [x2, y2], ...], ... }
    """
    path_images = os.path.join(directorio_raiz, "images")
    path_labels = os.path.join(directorio_raiz, "labels")

    if not os.path.exists(path_labels) or not os.path.exists(path_images):
        raise FileNotFoundError(
            f"No se encuentran images/ o labels/ en {directorio_raiz}")

    anotaciones = {}
    archivos_txt = [f for f in os.listdir(path_labels) if f.endswith(".txt")]

    for archivo_txt in sorted(archivos_txt):
        nombre_img = os.path.splitext(archivo_txt)[0] + ".jpg"
        ruta_img   = os.path.join(path_images, nombre_img)
        ruta_txt   = os.path.join(path_labels, archivo_txt)

        if not os.path.exists(ruta_img):
            continue  # imagen sin par de etiqueta

        # Dimensiones reales para desnormalizar las coordenadas
        with Image.open(ruta_img) as img:
            img_w, img_h = img.size

        puntos = []
        with open(ruta_txt, "r") as f:
            for linea in f:
                datos = linea.strip().split()
                if not datos:
                    continue
                clase = int(datos[0])
                if clase != 1:            # Solo clase 1 = cabeza
                    continue
                cx_norm = float(datos[1])
                cy_norm = float(datos[2])
                x_px = cx_norm * img_w    # desnormalizar a pixel
                y_px = cy_norm * img_h
                puntos.append([x_px, y_px])

        if puntos:  # solo imágenes con al menos una cabeza anotada
            anotaciones[nombre_img] = puntos

    return anotaciones

In [ ]:
# Anotaciones del conjunto de train+val
ann_train = cargar_puntos_cabeza_desde_yolo(train_root)
print(f"Imagenes con anotaciones (train+val): {len(ann_train)}")

# Anotaciones del conjunto de test fijo
ann_test = cargar_puntos_cabeza_desde_yolo(test_root)
print(f"Imagenes con anotaciones (test):      {len(ann_test)}")

# Comprobación de las primeras anotaciones
for idx, (img_id, puntos) in enumerate(ann_train.items()):
    if idx == 5: break
    pts_str = ",  ".join(f"({x:.1f},{y:.1f})" for x, y in puntos[:3])
    print(f"[{img_id}] {len(puntos)} cabezas -> [{pts_str} ...]")

## 3. Preprocesamiento

In [ ]:
#Establecer semilla para que las ejecuciones sean reproducibles
def set_seed(seed):
    random.seed(seed)           #Módulos estándar de Python
    np.random.seed(seed)        #Operaciones de NumPy
    torch.manual_seed(seed)     #CPU y GPU por defecto

    if torch.cuda.is_available():
       torch.cuda.manual_seed(seed)
       torch.cuda.manual_seed_all(seed)    #Todas las GPUs en el sistema

       #Modo de búsqueda donde cuDNN prueba múltiples algoritmos para una operación dada y selecciona el más rápido para el tamaño de entrada específico
       torch.backends.cudnn.benchmark = False #False = Evita la selección dinámica de algoritmos
       #Obliga a la biblioteca a utilizar únicamente algoritmos que son deterministas
       torch.backends.cudnn.deterministic = True

In [ ]:
@dataclass(frozen=True)
class TrainingConfig:

    img_size: int = 512         #Tamaño de entrada (H=W)
    batch_size: int = 16        #Muestras de entrenamiento procesadas simultaneamente
    num_epochs: int = 50        #Número de iteraciones completas sobre el conjunto de datos

    learning_rate: float = 1e-5 #Tasa de aprendizaje
    weight_decay: float = 5e-4  #Coeficiente de regularización L2
    max_norm: float = 10.0

    ### Sigma / densidad (Desviación que controla el radio de dispersión espacial aplicado a cada anotación para generar el mapa de densidad)
    sigma_base: float = 40.0     #Sigma de referencia como punto de partida
    adaptive_sigma: bool = False  #Activar sigma adaptativo

    # Parámetros de sigma adaptativo (distancia media a vecinos)
    adaptive_k: int = 3          # k: nº de vecinos
    adaptive_beta: float = 0.3   # beta: factor multiplicador para la distancia media (controla la dispersión del kernel Gaussiano)

    # Límites razonables de sigma en la resolución de salida para evitar extremos numéricos
    min_sigma_out: float = 1.5
    max_sigma_out: float = 10.0

    device: str = "cuda" if torch.cuda.is_available() else "cpu" #Hardware de cómputo

train_config = TrainingConfig()
device = train_config.device


**Utils para la generación del Ground Truth (mapas de densidad)**

In [ ]:
#Media y desviación previamente calculada
#mean = [0.44722986, 0.41087792, 0.39603603]
#std =  [0.26052581, 0.24937533, 0.2516997]

#Media y desviación propias de la normalización de VGG16
mean=[0.485, 0.456, 0.406]
std=[0.229, 0.224, 0.225]


def compute_adaptive_sigmas(points_xy: np.ndarray, k: int, beta: float) -> np.ndarray:
    """
    Calcula un sigma adaptativo por punto basado en distancia media a sus k vecinos más cercanos.

    points_xy: array shape (N, 2) en coordenadas (x, y).
    k: número de vecinos para estimar densidad local (típico 3).
    beta: factor multiplicador de la distancia media (control de dispersión).

    Devuelve:
        sigmas: array shape (N,) con sigma por punto (float).
    """
    #Si hay 0 o 1 punto, no se puede estimar vecindad: devolvemos sigma=1 por defecto
    if points_xy.shape[0] <= 1:
        return np.ones((points_xy.shape[0],), dtype=np.float32)

    #Cálculo de la matriz de distancias euclídeas NxN (coste O(N^2))
    diffs = points_xy[:, None, :] - points_xy[None, :, :]               # (N, N, 2)
    dists = np.sqrt(np.sum(diffs ** 2, axis=-1))                        # (N, N)

    #diagonal a infinito para que el punto no se considere a sí mismo como vecino
    np.fill_diagonal(dists, np.inf)

    #Se toman las k distancias más pequeñas por punto
    k_eff = min(k, points_xy.shape[0] - 1)                              # k efectivo (no puede exceder N-1)
    knn = np.partition(dists, kth=k_eff - 1, axis=1)[:, :k_eff]         # (N, k_eff)

    #Distancia media a k vecinos
    mean_knn = np.mean(knn, axis=1)                                     # (N,)

    #Sigma adaptativo: beta * distancia_media
    sigmas = beta * mean_knn

    #Si por algún caso extremo sigma sale 0 (puntos duplicados), se fuerza a mínimo > 0
    sigmas = np.maximum(sigmas, 1e-3).astype(np.float32)

    return sigmas


def create_density_map_adaptive(
    points: list,
    H: int,
    W: int,
    adaptive: bool,
    sigma_base: float,
    k: int,
    beta: float,
    min_sigma: float,
    max_sigma: float
) -> np.ndarray:
    """
    Crea un mapa de densidad (H, W) a partir de puntos
    - Si adaptive=False: usa sigma_base global
    - Si adaptive=True: usa sigma por punto calculado con kNN

    - Se usa un kernel Gaussiano por punto y se suma sobre el mapa
    - Se recorta ROI (Region of Interest) alrededor del punto para evitar coste O(H*W*N).
    """
    density = np.zeros((H, W), dtype=np.float32)

    #Si no hay puntos, devolvemos mapa vacío
    if points is None or len(points) == 0:
        return density

    pts = np.asarray(points, dtype=np.float32)  # shape (N,2) en (x,y)

    #Calculamos sigma por punto si es adaptativo, si no, todos igual
    if adaptive:
        sigmas = compute_adaptive_sigmas(pts, k=k, beta=beta)  # (N,)
    else:
        sigmas = np.full((pts.shape[0],), float(sigma_base), dtype=np.float32)

    #Límites para evitar kernels demasiado pequeños o grandes en el output
    sigmas = np.clip(sigmas, min_sigma, max_sigma)

    #Generamos un kernel Gaussiano por punto y lo acumulamos en una ventana local o región de interés (ROI)
    for (x, y), s in zip(pts, sigmas):
        #Redondeo al píxel más cercano
        xi = int(round(float(x)))
        yi = int(round(float(y)))

        #Ignorar puntos fuera de la imagen
        if xi < 0 or xi >= W or yi < 0 or yi >= H:
            continue

        #Radio del ROI = 3*sigma (más allá de 3 desviaciones estándar (3σ) la función Gaussiana ~= 0)
        radius = max(1, int(round(3.0 * float(s))))

        #Coordenadas del ROI recortadas a los bordes del mapa
        x1 = max(0, xi - radius)
        x2 = min(W, xi + radius + 1)
        y1 = max(0, yi - radius)
        y2 = min(H, yi + radius + 1)

        #Tamaño del kernel para el ROI actual
        kW = x2 - x1
        kH = y2 - y1

        #Parche/matriz del tamaño del ROI
        patch = np.zeros((kH, kW), dtype=np.float32)
        patch[yi - y1, xi - x1] = 1.0

        #Al patch se le aplica desenfoque gaussiano, OpenCV GaussianBlur
        patch = cv2.GaussianBlur(patch, ksize=(0, 0), sigmaX=float(s), sigmaY=float(s)) # ksize=(0,0) deja que OpenCV lo derive del sigma

        #Sumamos el patch al mapa de densidad global
        density[y1:y2, x1:x2] += patch

    return density


class CSRNet_FromGroundTruths(Dataset):
    """
    Clase tipo Dataset que define el acceso a cada elemento del conjunto de datos.

    images_dir: directorio con las imágenes
    annotations: diccionario de las coordenadas de los puntos de cada imagen (img_name: [[x1,y1], [x2,y2], ...])
    sigma: desviación estándar del kernel gaussiano
    image_size:  tamaño (height, width) al que las imagenes son reescaladas
    mean/std: valores de media y desiación para la normalización de las imágenes

    Devuelve:
      - img_tensor: imagen tensor (3, H, W)
      - gt_tensor: Ground Truth (mapa de densidad) (1, H, W)
      - pts: número de puntos (int)
    """

    def __init__(self, images_dir: str, annotations: dict = None):
        super().__init__()

        self.image_size = train_config.img_size
        self.sigma = train_config.sigma_base
        self.mean = mean
        self.std = std

        #Tranformaciones que se van a aplicar a la imagen
        self.transform = T.Compose([
            T.Resize((self.image_size, self.image_size)),
            T.ToTensor(),
            T.Normalize(mean=self.mean, std=self.std)
        ])

        # path de las imágenes
        images_glob = os.path.join(images_dir, "*.jpg")
        self.image_paths = sorted(glob(images_glob))
        if len(self.image_paths) == 0:
            raise ValueError(f"No images found in {images_dir} with pattern '*.jpg'")

        #Normalización de las claves de annotations para que coincidan con los nombres de las imágenes que se van a cargar
        # "data/train/0001.jpg" -> "0001.jpg"
        self.annotations = None
        if annotations is not None:
            new_ann = {}
            for k, v in annotations.items():
                new_ann[Path(k).name] = v
            self.annotations = new_ann

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):
        #Cargar la imagen original y recoger las dimensiones
        img_path = self.image_paths[index]
        img = Image.open(img_path).convert("RGB")
        orig_W, orig_H = img.size

        #Aplicar las transformaciones
        img_t = self.transform(img)

        img_name = Path(img_path).name
        if self.annotations is not None:
            pts_list = self.annotations.get(img_name, [])
            pts = len(pts_list) #Número de puntos en la imagen

            #En CSRNet la salida del modelo suele ser 1/8 del tamaño de la imagen
            #(evita problemas de downsampling, que ocurrirían si primero se hiciera la densidad al tamaño completo y luego la redujese a 1/8)
            out_H = self.image_size // 8
            out_W = self.image_size // 8

            #Escalar las coordenadas originales al nuevo tamaño
            x_out_scaled = out_W / float(orig_W)
            y_out_scaled = out_H / float(orig_H)
            scaled_pts_out = [(x * x_out_scaled, y * y_out_scaled) for (x, y) in pts_list]

            #Si sigma adaptativo = False:
            # se usasigma_base reescalado a la resolución de salida
            #Si sigma adaptativo = True:
            # se calcula sigma por punto a partir de distancias entre puntos en la resolución de salida
            avg_scale_out = 0.5 * (x_out_scaled + y_out_scaled)  # escala media

            sigma_base_out = float(train_config.sigma_base) * float(avg_scale_out)  # sigma de referencia en salida

            gt_resized = create_density_map_adaptive(
                points=scaled_pts_out,                       # puntos escalados a (out_W, out_H)
                H=out_H,
                W=out_W,
                adaptive=train_config.adaptive_sigma,
                sigma_base=sigma_base_out,
                k=train_config.adaptive_k,                   # nº vecinos para estimar densidad local
                beta=train_config.adaptive_beta,             # factor multiplicador de distancia media
                min_sigma=train_config.min_sigma_out,        # límite inferior
                max_sigma=train_config.max_sigma_out         # límite superior
            ).astype(np.float32)

        else:
            #Si no hay anotaciones, todo ceros en el mapa de densidad
            pts = 0
            out_H = max(1, self.image_size // 8)
            out_W = max(1, self.image_size // 8)
            gt_resized = np.zeros((out_H, out_W), dtype=np.float32)

        #La matriz del mapa de densidad se convierte a tensor (1, out_H, out_W) (En mapas de densidad un único canal de color)
        gt_tensor = T.ToTensor()(gt_resized.astype(np.float32))

        return img_t, gt_tensor, pts

Comprobacion del dataset con las anotaciones CrowdHuman

In [ ]:
dataset_checking = CSRNet_FromGroundTruths(
    images_dir=train_images_root,
    annotations=ann_train
)
check_dataset(dataset_checking, n=10)

## 4. Construcción del modelo

In [ ]:
def make_layers(config_architecture, in_channels=3, batch_norm=False, dilation=False):

    """
    Función para crear las capas de la red neuronal.
    - config_architecture: configuración de la arquitectura de capas a construir
    - in_channels: número de canales de entrada (3 para imágenes RGB)
    - batch_norm: si poner o no Batch Normalization
    - dilation: usar o no dilated convolutions

    """
    #Si dilation=True entonces el kernel de convolución se expande para capturar contexto más grande sin perder resolución espacial
    d_rate = 2 if dilation else 1
    layers = []
    for out_channels in config_architecture:
        if out_channels == 'M': #Añade capa de MaxPooling que reduce por dos
            layers += [nn.MaxPool2d(kernel_size=2, stride=2)]
        else:        #Si es un número, capa de Convolución
            conv2d = nn.Conv2d(in_channels, out_channels, kernel_size=3,
                               padding=d_rate, dilation=d_rate)
            if batch_norm:
                #Normalización si se indica, se coloca después de la convolución y antes de la activación (ReLU) para estabilidad del entrenamiento
                layers += [conv2d, nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True)]
            else:
                layers += [conv2d, nn.ReLU(inplace=True)]
            in_channels = out_channels #Para la próxima convolución, los canales de entrada serán los canales de salida de la capa actual
    return nn.Sequential(*layers) #Creación del bloque secuencial de capas

class CSRNet(nn.Module):

    """
    Clase del modelo CSRNet.
    """
    def __init__(self, load_weights=False):
        super(CSRNet, self).__init__()

        #Bloque de capas frontend típico de VGG
        self.frontend_feature = [64, 64, 'M',
                                128, 128, 'M',
                                256, 256, 256, 'M',
                                512, 512, 512]

        #Dilated backend standard de CSRNet
        self.backend_feature = [512, 512, 512, 256]

        self.frontend = make_layers(self.frontend_feature)
        self.backend = make_layers(self.backend_feature, in_channels=512, dilation=True)

        #Bloque de capas output típico de CSRNet
        self.output_layer = nn.Sequential(
            nn.Conv2d(256, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 1, kernel_size=1),
            #nn.ReLU(inplace=True)   #evita negativos en VERSION CSRNet Estable, sin ella versión CSRNet Pura
        )

        #Carga de los pesos preentrenados del modelo VGG16
        if not load_weights:
            vgg = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
            vgg_conv = [m for m in vgg.features if isinstance(m, nn.Conv2d)]
            self_conv = [m for m in self.frontend if isinstance(m, nn.Conv2d)]
            for s, v in zip(self_conv, vgg_conv):
                s.weight.data[:] = v.weight.data.clone()
                s.bias.data[:] = v.bias.data.clone()

        #Inicialización de las capas del backend y output (frontend se mantienen los preentrenados)
        self.initialize_weights(scope='backend_output')

        #Inicialización de la última capa (std=0.001 y bias=0)
        #Para que la red empiece prediciendo un mapa de densidad cercano a cero y evitar Loss inicial enorme e inestabilidad
        last = self.output_layer[-1]
        if isinstance(last, nn.Conv2d):
            nn.init.normal_(last.weight, std=1e-3)
            if last.bias is not None:
                nn.init.constant_(last.bias, 0.0)

    #Función de propagación hacia adelante en la red neuronal
    def forward(self, x):
        x = self.frontend(x)
        x = self.backend(x)
        x = self.output_layer(x)
        return x


    #Función de inicialización de pesos
    def initialize_weights(self, scope='all'):
        modules = []
        if scope == 'all':
            modules = self.modules()
        elif scope == 'backend_output':
            modules = list(self.backend.modules()) + list(self.output_layer.modules())
        for m in modules:
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)


## 5. Funciones de recogida de Logs

In [ ]:
class LogsFunction:

    def __init__(self, output_dir: str, num_epochs: int):
        self.output_dir = output_dir
        self.num_epochs = num_epochs

        log_files_count = len(glob(os.path.join(self.output_dir, "Log_*.txt")))

        self.output_file_dir = os.path.join(
            self.output_dir,
            f"Log_{log_files_count}_{datetime.now().strftime('%Y-%m-%d_%H-%M')}.txt"
        )

        self.create_log_file()

    def create_log_file(self):
        with open(self.output_file_dir, 'w', encoding="utf-8") as wr:
            wr.write(f"Fecha de creación: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

    def log_hyperparams(self, **config):
        #Bbloque de hiperparámetros para la trazabilidad
        with open(self.output_file_dir, 'a', encoding="utf-8") as wr:
            wr.write("\n======= Configuracion del entrenamiento =======\n")
            for key, value in config.items():
                wr.write(f"- {key}: {value}\n")
            wr.write("===============================================\n\n")

    def log_results(self, epoch: int, **metrics):
        with open(self.output_file_dir, 'a', encoding="utf-8") as writer:
            log = f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] [{epoch}/{self.num_epochs}] "
            for key, value in metrics.items():
                try:
                    value = float(value)
                    log += f"| {key}: {value:.6f} "
                except Exception:
                    log += f"| {key}: {value} "
            writer.write(log + "\n")

    def write(self, msg: str):
        #Mensaje general
        with open(self.output_file_dir, 'a', encoding="utf-8") as wr:
            timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            wr.write(f"[{timestamp}] {msg}\n")
        print(msg)

    def log_error(self, err: str):
        #Mensaje de error/warnings
        with open(self.output_file_dir, 'a', encoding="utf-8") as wr:
            timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            wr.write(f"[{timestamp}] {err}\n")
        print(err)

##6. Entrenamiento del modelo

In [ ]:
def count_metrics(p_gt, gt):

    #Paso de densidad a número de personas, p_gt y gt: (B,1,H,W)
    pred_counts = p_gt.view(p_gt.size(0), -1).sum(dim=1)  # p_gt.view(p_gt.size(0), -1): paso de (B,1,H,W) a (B, H*W), pred_counts tiene un valor por imagen
    gt_counts   = gt.view(gt.size(0), -1).sum(dim=1)

    #error absoluto y cuadrático de cada imagen
    abs_errs = torch.abs(pred_counts - gt_counts) # (B,)
    sq_errs  = (pred_counts - gt_counts)**2

    #Devuelve la suma de los errores del batch
    return abs_errs.sum().item(), sq_errs.sum().item()

def train_batch(model, optimizer, data, criterion):
    #modelo en modo entrenamiento
    model.train()

    img, gt, pt = data
    img = img.to(device)
    gt  = gt.to(device)

    if isinstance(pt, torch.Tensor):
        pt = pt.to(device)

    #Limpieza de gradientes (más eficiente en memoria que poner todos los gradientes a 0)
    optimizer.zero_grad(set_to_none=True)

    #Forward pass para obtener las predicciones de la imagen
    preds = model(img)   #(B,1,Hout,Wout)

    """
    # Debug: predicciones y GT antes de backprop
    with torch.no_grad():
        B = preds.size(0)
        pred_counts = preds.view(B, -1).sum(dim=1).cpu().numpy()
        gt_counts   = gt.view(B, -1).sum(dim=1).cpu().numpy()
        print("batch gt counts (first 8):", gt_counts[:8])
        print("batch pred counts (first 8):", pred_counts[:8])
    """
    #Cálculo del loss y backpropagation
    loss = criterion(preds, gt)
    loss.backward()

    #clip gradients para evitar gradientes explosivos
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=train_config.max_norm)

    #actualizar todos los pesos del modelo con los gradientes
    optimizer.step()

    """
    #Añadido para debug: pesos en la última capa
    grad_norm = 0.0
    for p in model.output_layer.parameters():
        if p.grad is not None: grad_norm += p.grad.data.norm(2).item()**2
    grad_norm = grad_norm**0.5
    print("last-layer grad norm:", grad_norm)

    with torch.no_grad():
        try:
            last_conv = model.output_layer[-1]
            if isinstance(last_conv, torch.nn.Conv2d):
                w = last_conv.weight.detach().cpu()
                b = last_conv.bias.detach().cpu() if last_conv.bias is not None else None
                print("after step - last conv - weight mean,std:", w.mean().item(), w.std().item(),
                      " bias mean,std:", None if b is None else (b.mean().item(), b.std().item()))
        except Exception as e:
            print("Error: no se puedo leer last_conv ", e)
    """

    #Métricas de conteo
    abs_err, sq_err = count_metrics(preds, gt)

    #pt_loss como MAE medio del batch
    pt_loss = abs_err / float(preds.size(0))

    #mover escalares fuera de los tensores
    loss_item = loss.item()
    #pt_loss_item = pt_loss.item()
    pt_loss_item = pt_loss

    #liberar memoria
    del loss, pt_loss
    del img, gt, preds

    return loss_item, pt_loss_item, abs_err, sq_err

def validate_batch(model, data, criterion):
    """
    Evalúa un batch de imágenes
    Devuelve:
      - loss_item: loss medio del batch
      - pt_loss_item: MAE medio por imagen del conteo
      - abs_err: suma de errores absolutos de conteo en el batch
      - sq_err: suma de errores cuadráticos de conteo en el batch
    """
    #Modelo en modo evaluación
    model.eval()

    img, gt, pt = data
    img = img.to(device)
    gt  = gt.to(device)
    if isinstance(pt, torch.Tensor):
        pt = pt.to(device)

    #No se hace torch.no_grad() porque se hace en la función principal antes de llamar a esta
    p_gt = model(img)

    loss = criterion(p_gt, gt)                  #Loss
    abs_err, sq_err = count_metrics(p_gt, gt)   #Métricas de conteo

    # pt_loss como MAE medio por imagen del batch
    pt_loss = abs_err / float(p_gt.size(0))

    #Convertir a escalares python antes de liberar tensores
    loss_item = float(loss.item())
    pt_loss_item = float(pt_loss)

    #Liberación de tensores de memoria (si no se acaba la RAM de la GPU en Google Colab)
    del loss, pt_loss, p_gt
    del img, gt, pt

    return loss_item, pt_loss_item, abs_err, sq_err


def visualize_density_maps(model, data, save_path=None):
    """
    Guarda imagen original, GT (mapa de densidad original) y predicción (mapa de desidad) en una imagen PNG
    """
    model.eval()
    img, gt, _ = data
    img = img.to(device)

    with torch.no_grad():
        pred = model(img)

    img_np = img[0].cpu().numpy().transpose(1,2,0)    # mueve el tensor a CPU para poder convertirlo en array NumPy, y ambia el orden de canales (C, H, W) → (H, W, C)
    img_np = img_np * np.array(std) + np.array(mean)  #desnormalizar la imagen
    img_np = np.clip(img_np, 0, 1)                    #Garantiza que los píxeles estén entre 0 y 1, para evitar: colores saturados, valores negativos

    gt_np = gt[0].squeeze().cpu().numpy()             # añadido .squeeze() para eliminar dimensiones de tamaño 1, (1, 1, H, W) → (H, W), convertir a Numpy para poder visualizarlos
    pred_np = pred[0].squeeze().cpu().numpy()

    fig, axs = plt.subplots(1, 3, figsize=(12, 4))

    axs[0].imshow(img_np)
    axs[0].set_title("Imagen")

    axs[1].imshow(gt_np, cmap='jet')
    axs[1].set_title(f"GT Density (count={gt_np.sum():.1f})")

    axs[2].imshow(pred_np, cmap='jet')
    axs[2].set_title(f"Prediction Density (count={pred_np.sum():.1f})")

    for a in axs:
        a.axis("off")

    if save_path:
        plt.savefig(save_path)
    plt.close()

**Funcion principal de entrenamiento**

In [ ]:
VIZ = 5

def CrowdCounting(
    train_images_root : str,
    test_images_root  : str,
    ann_train         : dict,
    ann_test          : dict,
    output_dir        : str,
    total_epochs      : int,
    model_pth         : Union[str, None] = None,
    viz_indices       : list = None
):
    """
    Funcion principal de entrenamiento CSRNet sobre CrowdHuman.

    Diferencias respecto a la version NWPU:
    - Split train/val con generator fijo (seed=42) igual que en YOLO.
    - Registro de tiempos de inferencia GPU por imagen.
    - L1Loss (mediana condicional) en lugar de MSELoss.
    - Visualizacion sobre imagenes fijas determinadas por viz_indices.
    """

    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(os.path.join(output_dir, "saved_weights"), exist_ok=True)
    os.makedirs(os.path.join(output_dir, "visualizations"), exist_ok=True)

    writer = SummaryWriter(output_dir)

    logger = LogsFunction(output_dir=output_dir, num_epochs=total_epochs)
    logger.log_hyperparams(
        batch_size=train_config.batch_size,
        num_epochs=train_config.num_epochs,
        learning_rate=train_config.learning_rate,
        max_norm=train_config.max_norm,
        sigma_base=train_config.sigma_base,
        adaptive_sigma=train_config.adaptive_sigma,
        adaptive_k=train_config.adaptive_k,
        adaptive_beta=train_config.adaptive_beta,
        max_sigma_out=train_config.max_sigma_out,
        min_sigma_out=train_config.min_sigma_out,
        img_size=train_config.img_size,
        loss="L1Loss",
        optimizer="Adam",
        scheduler="MultiStepLR(milestones=[25, 40], gamma=0.3)"
    )

    model = CSRNet().to(device)

    if model_pth:
        logger.write("[INFO] Cargando pesos del modelo.")
        try:
            model.load_state_dict(torch.load(model_pth), strict=True)
            logger.write("[INFO] Pesos cargados correctamente.")
        except Exception as e:
            logger.log_error(f"[ERROR] Al cargar pesos: {e}")
    else:
        logger.write("[INFO] Entrenando desde cero (sin pesos iniciales propios).")

    # L1Loss
    criterion = nn.L1Loss()

    optimizer = optim.Adam(
        model.parameters(),
        lr=train_config.learning_rate,
        weight_decay=train_config.weight_decay
    )
    scheduler = torch.optim.lr_scheduler.MultiStepLR(
        optimizer, milestones=[25, 40], gamma=0.3
    )

    # Dataset train+val
    dataset = CSRNet_FromGroundTruths(
        images_dir=train_images_root,
        annotations=ann_train
    )

    train_size = int(0.8 * len(dataset))
    val_size   = len(dataset) - train_size

    split_generator = torch.Generator().manual_seed(42)
    train_dataset, val_dataset = random_split(
        dataset, [train_size, val_size], generator=split_generator
    )

    trn_dl = DataLoader(train_dataset, batch_size=train_config.batch_size, shuffle=True)
    val_dl  = DataLoader(val_dataset,  batch_size=train_config.batch_size, shuffle=False)

    test_dataset = CSRNet_FromGroundTruths(
        images_dir=test_images_root,
        annotations=ann_test
    )
    test_dl = DataLoader(test_dataset, batch_size=1, shuffle=False)

    if viz_indices is None:
        viz_indices = list(range(min(5, len(val_dataset))))

    training_history = []
    best_loss = float("inf")

    for epoch in range(total_epochs):
        epoch_start = time.time()
        print(f"\nEpoch {epoch+1}/{total_epochs}")
        print("-" * 50)

        # Training
        total_tr_loss = total_tr_pts_loss = 0.0
        train_abs_err = train_sq_err = 0

        for data in tqdm(trn_dl, desc="Training: "):
            loss, pts_loss, abs_err, sq_err = train_batch(model, optimizer, data, criterion)
            total_tr_loss     += loss
            total_tr_pts_loss += pts_loss
            train_abs_err     += abs_err
            train_sq_err      += sq_err

        avg_tr_loss    = total_tr_loss / len(trn_dl)
        avg_tr_pt_loss = total_tr_pts_loss / len(trn_dl)
        train_mae      = train_abs_err / len(train_dataset)
        train_rmse     = sqrt(train_sq_err / len(train_dataset))

        # Validation
        total_val_loss = total_val_pts_loss = 0.0
        val_abs_err = val_sq_err = 0

        with torch.no_grad():
            for data in tqdm(val_dl, desc="Validating: "):
                val_loss, pts_loss, abs_err, sq_err = validate_batch(model, data, criterion)
                total_val_loss     += val_loss
                total_val_pts_loss += pts_loss
                val_abs_err        += abs_err
                val_sq_err         += sq_err

        avg_val_loss    = total_val_loss / len(val_dl)
        avg_val_pt_loss = total_val_pts_loss / len(val_dl)
        val_mae         = val_abs_err / len(val_dataset)
        val_rmse        = sqrt(val_sq_err / len(val_dataset))

        epoch_duration = time.time() - epoch_start
        current_lr     = optimizer.param_groups[0]["lr"]

        print(f"Train Loss: {avg_tr_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
        print(f"Train MAE:  {train_mae:.2f}   | Val MAE:  {val_mae:.2f}")
        print(f"Train RMSE: {train_rmse:.2f}  | Val RMSE: {val_rmse:.2f}")

        writer.add_scalar("Loss/train", avg_tr_loss,  epoch)
        writer.add_scalar("Loss/val",   avg_val_loss, epoch)
        writer.add_scalar("MAE/train",  train_mae,    epoch)
        writer.add_scalar("MAE/val",    val_mae,      epoch)
        writer.add_scalar("RMSE/train", train_rmse,   epoch)
        writer.add_scalar("RMSE/val",   val_rmse,     epoch)

        training_history.append({
            "Epoch":      epoch + 1,
            "LR":         current_lr,
            "Time_Sec":   round(epoch_duration, 2),
            "Train_Loss": avg_tr_loss,
            "Val_Loss":   avg_val_loss,
            "Train_MAE":  train_mae,
            "Val_MAE":    val_mae,
            "Train_RMSE": train_rmse,
            "Val_RMSE":   val_rmse,
        })

        if avg_val_loss < best_loss:
            best_loss  = avg_val_loss
            save_path  = os.path.join(output_dir, "saved_weights", f"Best_model_{epoch+1}.pth")
            torch.save(model.state_dict(), save_path)

        logger.log_results(
            epoch=epoch+1,
            tr_loss=avg_tr_loss,   val_loss=avg_val_loss,
            tr_pt_loss=avg_tr_pt_loss, val_pt_loss=avg_val_pt_loss,
            train_mae=train_mae,   train_rmse=train_rmse,
            val_mae=val_mae,       val_rmse=val_rmse,
        )

        scheduler.step()

        if epoch % VIZ == 0:
            for vi in viz_indices[:3]:  # max 3 imagenes por epoca
                try:
                    sample = val_dataset[vi]
                    img_t, gt_t, _ = sample
                    data_viz = (
                        img_t.unsqueeze(0),
                        gt_t.unsqueeze(0),
                        torch.tensor([0])
                    )
                    vis_path = os.path.join(
                        output_dir, "visualizations",
                        f"epoch_{epoch+1}_img{vi}.png"
                    )
                    visualize_density_maps(model, data_viz, save_path=vis_path)
                except Exception as e:
                    logger.log_error(f"[WARN] visualizacion idx {vi}: {e}")
            print(f"[+] Visualizaciones guardadas en {output_dir}/visualizations/")

    writer.close()

    # Evaluacion en TEST SET
    # Se usa el mejor modelo guardado durante el entrenamiento.
    logger.write("[INFO] Evaluando sobre test set...")
    best_weight = sorted(
        glob(os.path.join(output_dir, "saved_weights", "*.pth")),
        key=os.path.getmtime
    )[-1]
    model.load_state_dict(torch.load(best_weight))
    model.eval()

    test_abs_err  = 0
    test_sq_err   = 0
    inference_times = []   # tiempo GPU por imagen (ms)

    with torch.no_grad():
        for data in tqdm(test_dl, desc="Test evaluation: "):
            img_t, gt_t, _ = data
            img_t = img_t.to(device)
            gt_t  = gt_t.to(device)

            if device == "cuda":
                torch.cuda.synchronize()
            t_start = time.perf_counter()

            pred = model(img_t)

            if device == "cuda":
                torch.cuda.synchronize()
            t_end = time.perf_counter()

            inference_times.append((t_end - t_start) * 1000)  # ms

            abs_err, sq_err = count_metrics(pred, gt_t)
            test_abs_err += abs_err
            test_sq_err  += sq_err

            del img_t, gt_t, pred

    test_mae  = test_abs_err / len(test_dataset)
    test_rmse = sqrt(test_sq_err / len(test_dataset))
    t_median  = float(np.median(inference_times))
    t_mean    = float(np.mean(inference_times))

    logger.write(f"[RESULT] MAE_test:  {test_mae:.4f}")
    logger.write(f"[RESULT] RMSE_test: {test_rmse:.4f}")
    logger.write(f"[RESULT] Inference mediana: {t_median:.3f} ms/imagen")
    logger.write(f"[RESULT] Inference media:   {t_mean:.3f} ms/imagen")

    print(f"\n[TEST] MAE_test  = {test_mae:.4f}")
    print(f"[TEST] RMSE_test = {test_rmse:.4f}")
    print(f"[TEST] Inferencia mediana = {t_median:.3f} ms  |  media = {t_mean:.3f} ms")

    # Excel de metricas
    df = pd.DataFrame(training_history)
    df["Batch_Size"]       = train_config.batch_size
    df["Image_Size"]       = train_config.img_size
    df["Sigma"]            = train_config.sigma_base
    df["Min_sigma_out"]    = train_config.min_sigma_out
    df["Max_sigma_out"]    = train_config.max_sigma_out
    df["Learning_Rate"]    = train_config.learning_rate
    df["MAE_test"]         = test_mae
    df["RMSE_test"]        = test_rmse
    df["Inference_ms_med"] = t_median
    df["Inference_ms_avg"] = t_mean

    excel_path = os.path.join(output_dir, "metricas_training_validation.xlsx")
    try:
        df.to_excel(excel_path, index=False)
        print(f"[OK] Metricas guardadas en: {excel_path}")
    except Exception as e:
        csv_path = os.path.join(output_dir, "metricas_training_validation.csv")
        df.to_csv(csv_path, index=False)
        print(f"[INFO] Guardado como CSV: {csv_path}")

    return {
        "MAE_test":          test_mae,
        "RMSE_test":         test_rmse,
        "inference_ms_med":  t_median,
        "inference_ms_avg":  t_mean,
    }

In [ ]:
SEED = 28
set_seed(SEED)

VIZ_INDICES = [0, 50, 100, 200, 300]

output_dir  = f"output_csrnet_crowdhuman_seed{SEED}"
total_epoch = train_config.num_epochs
model_pth   = None

results = CrowdCounting(
    train_images_root = train_images_root,
    test_images_root  = test_images_root,
    ann_train         = ann_train,
    ann_test          = ann_test,
    output_dir        = output_dir,
    total_epochs      = total_epoch,
    model_pth         = model_pth,
    viz_indices       = VIZ_INDICES,
)

print("\n[RESUMEN FINAL]")
for k, v in results.items():
    print(f"  {k}: {v:.4f}")

In [ ]:
#Liberación de memoria
gc.collect()
torch.cuda.empty_cache()